# Mixed-Effects Regression: Milton + Helene Combined

**Objective**: Pool Milton county-level (N=21) and Helene cluster-level (N≈38) observations
into a single mixed-effects model to test whether socioeconomic factors predict
hurricane mobility disruption across two events.

**DVs (5)**:
1. Largest drop in within-region flow (%)
2. Largest drop in inflow (%)
3. Largest increase in outflow (%)
4. Total disruption — within (|drop| × recovery days)
5. Total disruption — inflow (|drop| × recovery days)

**Fixed effects (X)**:
- Hurricane category (Helene=4, Milton=5)
- Coastal indicator (1=coastal, 0=inland)
- Median household income
- % White population
- % No vehicle households
- Insurance coverage %
- NCHS urban-rural code
- Total population
- Population density (pop/km²)

**Random effect**: Hurricane (random intercept — accounts for baseline
difference between Cat 4 and Cat 5 impacts)

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import os
import warnings

import matplotlib.pyplot as plt
import matplotlib.cm as mcm
import matplotlib.colors as mcolors
import seaborn as sns

import statsmodels.api as sm
import statsmodels.formula.api as smf
from sklearn.preprocessing import StandardScaler
from scipy.stats import spearmanr, pearsonr

warnings.filterwarnings('ignore')

OUTPUT_DIR = '../results/mixed_model/'
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, 'figures'), exist_ok=True)
print('Setup complete.')

## 1. Load and Harmonize Data

In [ ]:
# ══════════════════════════════════════════════════════════════
# MILTON — county-level (N=21)
# ══════════════════════════════════════════════════════════════
m_drop_within = pd.read_csv('../results/milton/largest_drop_within.csv')
m_rec_within = pd.read_csv('../results/milton/recovery_within.csv')
m_drop_inflow = pd.read_csv('../results/milton/largest_drop_inflow.csv')
m_rec_inflow = pd.read_csv('../results/milton/recovery_inflow.csv')
m_outflow = pd.read_csv('../results/milton/outflow_increase.csv')

for d in [m_drop_within, m_rec_within, m_drop_inflow, m_rec_inflow, m_outflow]:
    d['GEOID'] = d['GEOID'].astype(int)

# Merge Milton
milton = m_drop_within[['GEOID', 'largest_drop']].rename(columns={'largest_drop': 'drop_within'})
milton = milton.merge(m_rec_within[['GEOID', 'recovery_days']].rename(
    columns={'recovery_days': 'recovery_within'}), on='GEOID', how='outer')
milton = milton.merge(m_drop_inflow[['GEOID', 'largest_drop']].rename(
    columns={'largest_drop': 'drop_inflow'}), on='GEOID', how='outer')
milton = milton.merge(m_rec_inflow[['GEOID', 'recovery_days']].rename(
    columns={'recovery_days': 'recovery_inflow'}), on='GEOID', how='outer')
milton = milton.merge(m_outflow[['GEOID', 'largest_increase']].rename(
    columns={'largest_increase': 'outflow_increase'}), on='GEOID', how='outer')

milton['hurricane'] = 'Milton'
milton['hurricane_cat'] = 5
milton['unit_type'] = 'county'
print(f'Milton: {len(milton)} counties')

# ══════════════════════════════════════════════════════════════
# HELENE — cluster-level
# ══════════════════════════════════════════════════════════════
h_within = pd.read_csv('../results/helene_cluster_analysis/cluster_results_within.csv')
h_inflow = pd.read_csv('../results/helene_cluster_analysis/cluster_results_inflow.csv')
h_outflow = pd.read_csv('../results/helene_cluster_analysis/cluster_results_outflow.csv')

helene = h_within[['cluster', 'n_counties', 'median_income', 'total_pop',
                    'mean_dist_to_track', 'cluster_pct_white', 'cluster_pct_bachelors',
                    'median_pct_no_vehicle', 'median_insurance', 'nchs_code',
                    'largest_drop', 'recovery_days']].rename(
    columns={'largest_drop': 'drop_within', 'recovery_days': 'recovery_within'})

helene = helene.merge(
    h_inflow[['cluster', 'largest_drop', 'recovery_days']].rename(
        columns={'largest_drop': 'drop_inflow', 'recovery_days': 'recovery_inflow'}),
    on='cluster', how='left')

# For outflow: use largest_drop as the disruption metric (not increase)
# Helene outflow shows DROPS (people couldn't leave), not increases
helene = helene.merge(
    h_outflow[['cluster', 'largest_drop']].rename(
        columns={'largest_drop': 'outflow_drop'}),
    on='cluster', how='left')

helene['hurricane'] = 'Helene'
helene['hurricane_cat'] = 4
helene['unit_type'] = 'cluster'
print(f'Helene: {len(helene)} clusters')

In [ ]:
# ══════════════════════════════════════════════════════════════
# ADD SOCIOECONOMIC + GEOGRAPHIC FEATURES
# ══════════════════════════════════════════════════════════════

# ── ACS data ──
acs = pd.read_csv('acs_socioeconomic_v2.csv')
acs['GEOID'] = acs['GEOID'].astype(int)

# ── NCHS urban-rural ──
nchs = pd.read_csv('../data/NCHS Urban-Rural Classification Scheme for Counties.csv', encoding='utf-8-sig')
nchs['GEOID'] = nchs['Location'].astype(int)
nchs['nchs_code'] = nchs['2023 Code'].str.extract(r'(\d)').astype(int)

# ── County geometry for coastal + pop density ──
county_shp = gpd.read_file('./../../hurricane_oct/data/county_geo/tl_2023_us_county/tl_2023_us_county.shp')
county_shp['GEOID'] = county_shp['GEOID'].astype(int)
county_shp['ALAND_km2'] = county_shp['ALAND'].astype(float) / 1e6  # m² → km²
county_shp['AWATER_km2'] = county_shp['AWATER'].astype(float) / 1e6

# ── Coastal indicator ──
# Simple heuristic: county is coastal if water area > 10% of total area
# or if it's in a known coastal FIPS list
county_shp['water_pct'] = county_shp['AWATER_km2'] / (county_shp['ALAND_km2'] + county_shp['AWATER_km2']) * 100

# Load NOAA coastal county list if available, otherwise use water % threshold
# NOAA defines 672 coastal counties — we'll approximate with water_pct > 15%
county_shp['is_coastal'] = (county_shp['water_pct'] > 15).astype(int)
print(f'Coastal counties (water > 15%): {county_shp["is_coastal"].sum()} / {len(county_shp)}')

# ── Distance to track for Milton ──
m_dist = pd.read_csv('../results/spatial_diagnostics_milton/county_distance_to_track.csv')
m_dist['GEOID'] = m_dist['GEOID'].astype(int)

# ── Merge features to MILTON ──
geo_idx = pd.read_csv('geoid_idx_names.csv')
geo_idx['GEOID'] = geo_idx['GEOID'].astype(int)

milton = milton.merge(acs[['GEOID', 'total_population', 'median_household_income',
                           'pct_no_vehicle', 'insurance_coverage_pct',
                           'pct_white', 'pct_bachelors_plus']], on='GEOID', how='left')
milton = milton.merge(nchs[['GEOID', 'nchs_code']], on='GEOID', how='left')
milton = milton.merge(m_dist[['GEOID', 'dist_to_track_mi']], on='GEOID', how='left')
milton = milton.merge(county_shp[['GEOID', 'ALAND_km2', 'is_coastal']], on='GEOID', how='left')
milton = milton.merge(geo_idx[['GEOID', 'NAME']], on='GEOID', how='left')

# Population density
milton['pop_density'] = milton['total_population'] / milton['ALAND_km2']

# Compute total disruption
milton['total_disruption_within'] = milton['recovery_within'] * milton['drop_within'].abs()
milton['total_disruption_inflow'] = milton['recovery_inflow'] * milton['drop_inflow'].abs()

print(f'Milton features merged: {len(milton)} rows')
print(f'Milton coastal: {milton["is_coastal"].sum()} / {len(milton)}')

In [ ]:
# ── Harmonize Helene cluster features ──
# Helene already has cluster-level features from clustering notebook
# Need to add: coastal, pop_density, ALAND

# Load cluster assignments to get county→cluster mapping
h_assign = pd.read_csv('../results/helene_clustering/county_cluster_assignments.csv')
h_assign['GEOID'] = h_assign['GEOID'].astype(int)

# Merge county-level ALAND and coastal
h_assign = h_assign.merge(county_shp[['GEOID', 'ALAND_km2', 'is_coastal']], on='GEOID', how='left')

# Compute cluster-level: total ALAND, coastal fraction
cluster_geo = h_assign.groupby('cluster').agg(
    total_aland_km2=('ALAND_km2', 'sum'),
    coastal_frac=('is_coastal', 'mean'),
).reset_index()

# Coastal cluster if >50% of counties are coastal
cluster_geo['is_coastal'] = (cluster_geo['coastal_frac'] > 0.5).astype(int)

helene = helene.merge(cluster_geo, on='cluster', how='left')
helene['pop_density'] = helene['total_pop'] / helene['total_aland_km2']

# Compute total disruption
helene['total_disruption_within'] = helene['recovery_within'] * helene['drop_within'].abs()
helene['total_disruption_inflow'] = helene['recovery_inflow'] * helene['drop_inflow'].abs()

print(f'Helene features merged: {len(helene)} clusters')
print(f'Helene coastal: {helene["is_coastal"].sum()} / {len(helene)}')

In [ ]:
# ══════════════════════════════════════════════════════════════
# STACK INTO ONE DATAFRAME
# ══════════════════════════════════════════════════════════════

# Harmonize column names
milton_std = milton.rename(columns={
    'median_household_income': 'income',
    'pct_white': 'pct_white',
    'pct_no_vehicle': 'pct_no_vehicle',
    'insurance_coverage_pct': 'insurance_pct',
    'dist_to_track_mi': 'dist_to_track',
})
milton_std['obs_id'] = milton_std['GEOID'].astype(str)

helene_std = helene.rename(columns={
    'median_income': 'income',
    'cluster_pct_white': 'pct_white',
    'median_pct_no_vehicle': 'pct_no_vehicle',
    'median_insurance': 'insurance_pct',
    'mean_dist_to_track': 'dist_to_track',
})
helene_std['obs_id'] = 'H_' + helene_std['cluster'].astype(str)

# Common columns for stacking
COMMON_COLS = [
    'obs_id', 'hurricane', 'hurricane_cat', 'unit_type',
    # DVs
    'drop_within', 'recovery_within', 'total_disruption_within',
    'drop_inflow', 'recovery_inflow', 'total_disruption_inflow',
    'outflow_increase',  # Milton only
    # IVs
    'income', 'pct_white', 'pct_no_vehicle', 'insurance_pct',
    'nchs_code', 'total_pop', 'pop_density',
    'is_coastal', 'dist_to_track',
]

# Add outflow_increase as NaN for Helene (different metric — drop vs increase)
if 'outflow_increase' not in helene_std.columns:
    helene_std['outflow_increase'] = np.nan

# Ensure all columns exist
for col in COMMON_COLS:
    if col not in milton_std.columns:
        milton_std[col] = np.nan
    if col not in helene_std.columns:
        helene_std[col] = np.nan

df = pd.concat([milton_std[COMMON_COLS], helene_std[COMMON_COLS]], ignore_index=True)

# Create binary urban-rural
df['is_nonmetro'] = (df['nchs_code'] >= 4).astype(float)

# Log population and pop density (highly skewed)
df['log_pop'] = np.log10(df['total_pop'])
df['log_density'] = np.log10(df['pop_density'].clip(lower=1))

print(f'\nCombined dataset: {len(df)} observations')
print(f'  Milton: {(df["hurricane"]=="Milton").sum()}')
print(f'  Helene: {(df["hurricane"]=="Helene").sum()}')
print(f'\nMissing values per column:')
print(df.isnull().sum()[df.isnull().sum() > 0])

df.to_csv(os.path.join(OUTPUT_DIR, 'combined_dataset.csv'), index=False)
print(f'\nSaved to {OUTPUT_DIR}/combined_dataset.csv')

## 2. Exploratory: Distribution Comparison

In [ ]:
# ── Compare feature distributions between hurricanes ──
compare_vars = ['income', 'pct_white', 'pct_no_vehicle', 'insurance_pct',
                'pop_density', 'is_coastal', 'nchs_code', 'dist_to_track']

fig, axes = plt.subplots(2, 4, figsize=(20, 8))

for ax, var in zip(axes.flat, compare_vars):
    for hrc, color in [('Milton', 'steelblue'), ('Helene', 'coral')]:
        vals = df[df['hurricane'] == hrc][var].dropna()
        if var in ['is_coastal', 'nchs_code']:
            counts = vals.value_counts().sort_index()
            offset = -0.15 if hrc == 'Milton' else 0.15
            ax.bar(counts.index + offset, counts.values, width=0.3,
                   color=color, alpha=0.7, label=hrc, edgecolor='black')
        else:
            ax.hist(vals, bins=10, alpha=0.5, color=color, label=hrc, edgecolor='black')
    ax.set_title(var, fontsize=11, fontweight='bold')
    ax.legend(fontsize=8)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Feature Distributions: Milton (N=21) vs Helene (N=38)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'figures', 'feature_distributions_comparison.png'),
            dpi=150, bbox_inches='tight')
plt.show()

## 3. Mixed-Effects Models

Use `statsmodels.MixedLM` with hurricane as random intercept.
This accounts for the baseline difference between Cat 4 (Helene) and Cat 5 (Milton)
while estimating common slopes for socioeconomic predictors.

In [ ]:
# ── Define DVs and IV sets ──
DV_CONFIGS = {
    'drop_within': 'Largest Drop — Within (%)',
    'drop_inflow': 'Largest Drop — Inflow (%)',
    'total_disruption_within': 'Total Disruption — Within',
    'total_disruption_inflow': 'Total Disruption — Inflow',
}

# Standardize continuous IVs for comparable coefficients
IV_CONTINUOUS = ['income', 'pct_white', 'pct_no_vehicle', 'insurance_pct',
                 'log_pop', 'log_density', 'dist_to_track']
IV_BINARY = ['is_coastal', 'is_nonmetro']
ALL_IVS = IV_CONTINUOUS + IV_BINARY

# Standardize
scaler = StandardScaler()
df_model = df.copy()
for col in IV_CONTINUOUS:
    valid = df_model[col].notna()
    if valid.sum() > 0:
        df_model.loc[valid, col + '_z'] = scaler.fit_transform(
            df_model.loc[valid, [col]]).flatten()

IV_Z = [c + '_z' for c in IV_CONTINUOUS] + IV_BINARY
print(f'Standardized IVs: {IV_Z}')

In [ ]:
# ══════════════════════════════════════════════════════════════
# RUN MIXED-EFFECTS MODELS
# ══════════════════════════════════════════════════════════════

model_results = {}

for dv_col, dv_label in DV_CONFIGS.items():
    print(f"\n{'='*70}")
    print(f'DV: {dv_label}')
    print(f"{'='*70}")
    
    # Drop rows with missing DV or IVs
    cols_needed = [dv_col, 'hurricane'] + IV_Z
    df_sub = df_model.dropna(subset=[c for c in cols_needed if c in df_model.columns]).copy()
    
    if len(df_sub) < 10:
        print(f'  ⚠ Only {len(df_sub)} observations — skipping')
        continue
    
    print(f'  N = {len(df_sub)} (Milton: {(df_sub["hurricane"]=="Milton").sum()}, '
          f'Helene: {(df_sub["hurricane"]=="Helene").sum()})')
    
    # ── Model 1: Pooled OLS (no random effects) ──
    print(f'\n  --- Model 1: Pooled OLS (hurricane as fixed effect) ---')
    formula_ols = f'{dv_col} ~ hurricane_cat + ' + ' + '.join([c for c in IV_Z if c in df_sub.columns])
    try:
        X_ols = df_sub[['hurricane_cat'] + [c for c in IV_Z if c in df_sub.columns]].copy()
        X_ols = sm.add_constant(X_ols)
        m_ols = sm.OLS(df_sub[dv_col], X_ols).fit()
        print(f'  R² = {m_ols.rsquared:.4f}, Adj.R² = {m_ols.rsquared_adj:.4f}, '
              f'F p = {m_ols.f_pvalue:.4e}')
        print(f'  {"Variable":<25} {"Coef":>8} {"p":>8} {"Sig":>5}')
        print(f'  {"-"*46}')
        for var in m_ols.params.index:
            sig = '**' if m_ols.pvalues[var] < 0.05 else '*' if m_ols.pvalues[var] < 0.1 else ''
            print(f'  {var:<25} {m_ols.params[var]:>8.3f} {m_ols.pvalues[var]:>8.4f} {sig:>5}')
    except Exception as e:
        print(f'  OLS failed: {e}')
        m_ols = None
    
    # ── Model 2: Mixed-effects (hurricane as random intercept) ──
    print(f'\n  --- Model 2: Mixed-effects (hurricane = random intercept) ---')
    try:
        iv_cols = [c for c in IV_Z if c in df_sub.columns]
        X_me = df_sub[iv_cols].copy()
        X_me = sm.add_constant(X_me)
        
        m_mixed = sm.MixedLM(
            df_sub[dv_col],
            X_me,
            groups=df_sub['hurricane']
        ).fit(reml=True)
        
        print(m_mixed.summary())
        model_results[dv_col] = {'ols': m_ols, 'mixed': m_mixed, 'n': len(df_sub)}
    except Exception as e:
        print(f'  Mixed model failed: {e}')
        import traceback
        traceback.print_exc()
        model_results[dv_col] = {'ols': m_ols, 'mixed': None, 'n': len(df_sub)}

## 4. Summary Table

In [ ]:
# ── Extract significant results across all DVs ──
print('='*80)
print('SIGNIFICANT RESULTS SUMMARY (p < 0.10)')
print('='*80)

sig_rows = []

for dv_col, dv_label in DV_CONFIGS.items():
    if dv_col not in model_results:
        continue
    res = model_results[dv_col]
    
    # Check mixed model
    m = res.get('mixed')
    if m is not None:
        for var in m.fe_params.index:
            p = m.pvalues[var]
            if p < 0.10 and var != 'const':
                sig_rows.append({
                    'DV': dv_label,
                    'IV': var,
                    'Model': 'Mixed',
                    'Coef': round(m.fe_params[var], 4),
                    'p': round(p, 4),
                    'N': res['n'],
                    'Sig': '**' if p < 0.05 else '*',
                })
    
    # Check OLS
    m_ols = res.get('ols')
    if m_ols is not None:
        for var in m_ols.params.index:
            p = m_ols.pvalues[var]
            if p < 0.10 and var != 'const':
                sig_rows.append({
                    'DV': dv_label,
                    'IV': var,
                    'Model': 'OLS',
                    'Coef': round(m_ols.params[var], 4),
                    'p': round(p, 4),
                    'N': res['n'],
                    'Sig': '**' if p < 0.05 else '*',
                })

if sig_rows:
    sig_df = pd.DataFrame(sig_rows)
    display(sig_df.sort_values(['DV', 'p']))
    sig_df.to_csv(os.path.join(OUTPUT_DIR, 'significant_results.csv'), index=False)
else:
    print('No significant results at p < 0.10')

## 5. Coefficient Plot

In [ ]:
# ── Forest plot: mixed model coefficients across DVs ──
fig, axes = plt.subplots(1, len(DV_CONFIGS), figsize=(6*len(DV_CONFIGS), 8),
                          sharey=True)
if len(DV_CONFIGS) == 1:
    axes = [axes]

for ax, (dv_col, dv_label) in zip(axes, DV_CONFIGS.items()):
    if dv_col not in model_results or model_results[dv_col]['mixed'] is None:
        ax.set_title(f'{dv_label}\n(model failed)', fontsize=10)
        continue
    
    m = model_results[dv_col]['mixed']
    params = m.fe_params.drop('const', errors='ignore')
    ci = m.conf_int().drop('const', errors='ignore')
    pvals = m.pvalues.drop('const', errors='ignore')
    
    y_pos = range(len(params))
    colors = ['darkred' if p < 0.05 else 'darkorange' if p < 0.1 else 'gray'
              for p in pvals]
    
    ax.barh(y_pos, params.values, color=colors, alpha=0.7, edgecolor='black')
    ax.errorbar(params.values, y_pos,
                xerr=[params.values - ci.iloc[:, 0].values,
                      ci.iloc[:, 1].values - params.values],
                fmt='none', color='black', capsize=3)
    ax.axvline(0, color='black', ls='--', lw=1)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(params.index, fontsize=9)
    ax.set_title(dv_label, fontsize=10, fontweight='bold')
    ax.set_xlabel('Coefficient (standardized)', fontsize=9)
    ax.grid(axis='x', alpha=0.3)
    
    # Annotate p-values
    for i, (coef, p) in enumerate(zip(params.values, pvals)):
        sig = '**' if p < 0.05 else '*' if p < 0.1 else ''
        if sig:
            ax.text(coef, i, f' p={p:.3f}{sig}', va='center', fontsize=7, fontweight='bold')

# Legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='darkred', alpha=0.7, label='p < 0.05'),
                   Patch(facecolor='darkorange', alpha=0.7, label='p < 0.10'),
                   Patch(facecolor='gray', alpha=0.7, label='n.s.')]
fig.legend(handles=legend_elements, loc='lower center', ncol=3, fontsize=10,
           bbox_to_anchor=(0.5, -0.02))

fig.suptitle('Mixed-Effects Model Coefficients (Hurricane = Random Intercept)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'figures', 'coefficient_forest_plot.png'),
            dpi=150, bbox_inches='tight')
plt.show()

## 6. Bivariate Correlations (Pooled)

In [ ]:
# ── Correlation heatmap: IVs vs DVs ──
dv_cols = list(DV_CONFIGS.keys())
iv_cols_raw = ['income', 'pct_white', 'pct_no_vehicle', 'insurance_pct',
               'log_pop', 'log_density', 'dist_to_track', 'is_coastal', 'is_nonmetro']

# Spearman correlations
corr_rows = []
for iv in iv_cols_raw:
    for dv in dv_cols:
        valid = df[[iv, dv]].dropna()
        if len(valid) >= 5:
            rho, p = spearmanr(valid[iv], valid[dv])
            corr_rows.append({'IV': iv, 'DV': DV_CONFIGS[dv], 'rho': rho, 'p': p})

corr_df = pd.DataFrame(corr_rows)
corr_pivot = corr_df.pivot(index='IV', columns='DV', values='rho')
p_pivot = corr_df.pivot(index='IV', columns='DV', values='p')

# Annotate with significance stars
annot = corr_pivot.copy()
for iv in annot.index:
    for dv in annot.columns:
        r = corr_pivot.loc[iv, dv]
        p = p_pivot.loc[iv, dv]
        star = '**' if p < 0.05 else '*' if p < 0.1 else ''
        annot.loc[iv, dv] = f'{r:.2f}{star}'

fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(corr_pivot.astype(float), annot=annot, fmt='', cmap='RdBu_r',
            center=0, vmin=-0.6, vmax=0.6, linewidths=0.5, ax=ax)
ax.set_title('Spearman Correlations: IVs vs DVs (pooled Milton + Helene)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'figures', 'correlation_heatmap.png'),
            dpi=150, bbox_inches='tight')
plt.show()

print('\nSignificant correlations (p < 0.10):')
sig_corr = corr_df[corr_df['p'] < 0.10].sort_values('p')
print(sig_corr.to_string(index=False))

## 7. Interpretation

**Mixed-effects model**:
- Random intercept for `hurricane` captures the baseline magnitude difference
  between Cat 4 (Helene) and Cat 5 (Milton)
- Fixed effects estimate the **common** relationship between socioeconomic
  factors and disruption, pooling evidence across both events
- With only 2 groups, the random intercept is heavily shrunk —
  effectively similar to a fixed-effect dummy for hurricane

**Caveats**:
- Milton uses county-level data (N=21), Helene uses cluster-level (N≈38)
- The unit of analysis differs → heteroscedasticity possible
- Helene clusters aggregate ~7 counties each → smoother estimates
- Results should be interpreted as hypothesis-generating, not confirmatory